# 🧪 Phase 2: Advanced Feature Engineering

**New features beyond original PSD:**

| Feature Group | Features |
|---|---|
| **Frequency Power** | Delta, Theta, Alpha, Beta, Gamma PSD |
| **Ratios** | Theta/Beta, Alpha/Beta, Gamma/Beta |
| **Connectivity** | Phase Locking Value (PLV), Coherence |
| **Complexity** | Sample Entropy, Permutation Entropy, Fractal Dimension |
| **Hjorth** | Activity, Mobility, Complexity |
| **Asymmetry** | Frontal Alpha Asymmetry (FAA), Hemispheric Theta Asymmetry |

All features are computed **per epoch** and saved to a single `comprehensive_features.csv`.

In [ ]:
import os
import gc
import mne
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from scipy.signal import welch, coherence
from scipy.stats import entropy as scipy_entropy
from sklearn.preprocessing import StandardScaler
from itertools import combinations

mne.set_log_level('WARNING')

OUTPUT_PATH = Path(os.getcwd()) / 'processed'
subjects    = sorted([p.stem.split('_task')[0] for p in OUTPUT_PATH.glob('*_task-med1breath_epochs.fif')])
tasks       = ['med1breath', 'med2', 'think1', 'think2']

SFREQ = 128
FREQ_BANDS = {
    'Delta': (0.5, 4),
    'Theta': (4,   8),
    'Alpha': (8,  12),
    'Beta':  (12, 30),
    'Gamma': (30, 40),
}

print(f'✅ Found {len(subjects)} subjects')
print(f'   Example: {subjects[:3]}')

## 🔌 Feature Block 1: Phase Locking Value (PLV)

PLV measures **synchrony** between two EEG channels — how locked their phases are in a frequency band. High PLV = channels are communicating.

$$PLV = \left|\frac{1}{N}\sum_{t=1}^{N} e^{i(\phi_1(t) - \phi_2(t))}\right|$$

In [ ]:
from scipy.signal import hilbert, butter, filtfilt

def bandpass(signal, lo, hi, fs):
    """Zero-phase bandpass filter."""
    nyq = fs / 2
    b, a = butter(4, [lo / nyq, hi / nyq], btype='band')
    return filtfilt(b, a, signal)


def compute_plv(signal1, signal2, lo, hi, fs):
    """
    Phase Locking Value between two signals in a frequency band.
    Returns scalar in [0, 1].
    """
    s1 = bandpass(signal1, lo, hi, fs)
    s2 = bandpass(signal2, lo, hi, fs)
    phase_diff = np.angle(hilbert(s1)) - np.angle(hilbert(s2))
    return np.abs(np.mean(np.exp(1j * phase_diff)))


def plv_features(epoch_data, ch_names, fs, bands=None, pairs=None):
    """
    Compute PLV for selected channel pairs and frequency bands.
    epoch_data: (n_channels, n_times)
    Returns dict of feature_name -> value.
    """
    if bands is None:
        bands = {'Alpha': (8, 12), 'Theta': (4, 8), 'Beta': (12, 30)}

    # Frontal-parietal and interhemispheric pairs of interest
    if pairs is None:
        all_idx  = list(range(min(8, len(ch_names))))  # limit to first 8 channels
        pairs    = list(combinations(all_idx, 2))[:10]  # top 10 pairs

    feats = {}
    for band, (lo, hi) in bands.items():
        plv_vals = []
        for i, j in pairs:
            plv = compute_plv(epoch_data[i], epoch_data[j], lo, hi, fs)
            plv_vals.append(plv)
        feats[f'PLV_{band}_mean'] = np.mean(plv_vals)
        feats[f'PLV_{band}_max']  = np.max(plv_vals)
        feats[f'PLV_{band}_std']  = np.std(plv_vals)
    return feats

print('✅ PLV functions defined.')

## 🌊 Feature Block 2: Spectral Coherence

Coherence measures frequency-specific **correlation** between channels — more robust to noise than PLV.

In [ ]:
def coherence_features(epoch_data, fs, bands=None, n_pairs=5):
    """
    Mean coherence in frequency bands for top N channel pairs.
    """
    if bands is None:
        bands = {'Alpha': (8, 12), 'Theta': (4, 8), 'Beta': (12, 30)}

    n_ch   = min(8, epoch_data.shape[0])
    pairs  = list(combinations(range(n_ch), 2))[:n_pairs]
    feats  = {}

    for band, (lo, hi) in bands.items():
        coh_vals = []
        for i, j in pairs:
            f, Cxy = coherence(epoch_data[i], epoch_data[j], fs=fs, nperseg=fs)
            mask = (f >= lo) & (f <= hi)
            if mask.sum() > 0:
                coh_vals.append(Cxy[mask].mean())
        if coh_vals:
            feats[f'Coherence_{band}_mean'] = np.mean(coh_vals)
            feats[f'Coherence_{band}_max']  = np.max(coh_vals)
    return feats

print('✅ Coherence functions defined.')

## 🔬 Feature Block 3: Non-linear Complexity Features

Meditation shifts the brain toward more **complex, high-entropy** dynamics in theta/alpha bands.

In [ ]:
def sample_entropy(signal, m=2, r_factor=0.2):
    """
    Sample Entropy — measures unpredictability of a time series.
    High SampEn = more complex/irregular signal.
    m: embedding dimension | r: tolerance (fraction of std)
    """
    r    = r_factor * np.std(signal)
    N    = len(signal)
    B, A = 0, 0
    templates_m   = np.array([signal[i:i+m]   for i in range(N - m)])
    templates_m1  = np.array([signal[i:i+m+1] for i in range(N - m)])

    for i in range(N - m):
        # Count template matches of length m and m+1
        diffs_m  = np.max(np.abs(templates_m   - templates_m[i]),   axis=1)
        diffs_m1 = np.max(np.abs(templates_m1  - templates_m1[i]), axis=1)
        B += np.sum(diffs_m  < r) - 1   # exclude self-match
        A += np.sum(diffs_m1 < r) - 1

    if B == 0 or A == 0:
        return 0.0
    return -np.log(A / B)


def permutation_entropy(signal, order=3, delay=1):
    """
    Permutation Entropy — captures ordinal patterns.
    Low PE = more regular/predictable (often in deep states).
    """
    n = len(signal)
    permutations = {}
    for i in range(n - (order - 1) * delay):
        window = signal[i:i + order * delay:delay]
        key    = tuple(np.argsort(window))
        permutations[key] = permutations.get(key, 0) + 1
    counts = np.array(list(permutations.values()), dtype=float)
    probs  = counts / counts.sum()
    return -np.sum(probs * np.log2(probs + 1e-10))


def higuchi_fd(signal, kmax=5):
    """
    Higuchi Fractal Dimension — measures signal self-similarity.
    HFD ~ 1.5 for noise, higher = more complex.
    """
    N   = len(signal)
    L   = []
    x   = np.array(signal)
    for k in range(1, kmax + 1):
        Lk = []
        for m in range(1, k + 1):
            indices = np.arange(m - 1, N, k)
            Lmk = np.sum(np.abs(np.diff(x[indices]))) * (N - 1) / (k * len(indices))
            Lk.append(Lmk)
        L.append(np.mean(Lk))
    ks   = np.arange(1, kmax + 1)
    logs = np.log(L)
    fd   = np.polyfit(np.log(ks), logs, 1)[0]
    return fd


def complexity_features(epoch_data, short_signal=True):
    """
    Compute Sample Entropy, Permutation Entropy, Higuchi FD
    averaged across all EEG channels.
    """
    n_ch = min(8, epoch_data.shape[0])  # limit channels for speed
    sampen_vals, permen_vals, hfd_vals = [], [], []

    for ch in range(n_ch):
        sig = epoch_data[ch]
        if short_signal:
            sig = sig[:128]  # use first 1s for speed

        sampen_vals.append(sample_entropy(sig))
        permen_vals.append(permutation_entropy(sig))
        hfd_vals.append(higuchi_fd(sig))

    return {
        'SampleEntropy_mean':  np.mean(sampen_vals),
        'PermEntropy_mean':    np.mean(permen_vals),
        'HiguchiFD_mean':      np.mean(hfd_vals),
        'SampleEntropy_std':   np.std(sampen_vals),
        'PermEntropy_std':     np.std(permen_vals),
    }

print('✅ Complexity (entropy/fractal) functions defined.')

## 📐 Feature Block 4: Hjorth Parameters

Hjorth parameters capture **activity, mobility, and complexity** of the EEG signal — computationally cheap but surprisingly powerful.

In [ ]:
def hjorth_parameters(signal):
    """
    Hjorth Activity: signal variance (power)
    Hjorth Mobility: std of 1st derivative / std of signal
    Hjorth Complexity: mobility of 1st derivative / mobility of signal
    """
    activity   = np.var(signal)
    diff1      = np.diff(signal)
    diff2      = np.diff(diff1)

    mobility   = np.sqrt(np.var(diff1) / (activity + 1e-10))
    complexity = np.sqrt(np.var(diff2) / (np.var(diff1) + 1e-10)) / (mobility + 1e-10)

    return activity, mobility, complexity


def hjorth_features(epoch_data):
    """Compute Hjorth parameters averaged across channels."""
    acts, mobs, comps = [], [], []
    for ch in range(epoch_data.shape[0]):
        a, m, c = hjorth_parameters(epoch_data[ch])
        acts.append(a)
        mobs.append(m)
        comps.append(c)
    return {
        'Hjorth_Activity_mean':    np.mean(acts),
        'Hjorth_Mobility_mean':    np.mean(mobs),
        'Hjorth_Complexity_mean':  np.mean(comps),
        'Hjorth_Activity_std':     np.std(acts),
        'Hjorth_Mobility_std':     np.std(mobs),
    }

print('✅ Hjorth parameter functions defined.')

## 🧠 Feature Block 5: Frontal Alpha Asymmetry (FAA)

FAA is one of the **most well-validated EEG biomarkers** for emotional and meditative states.

$$FAA = \ln(\alpha_{F4}) - \ln(\alpha_{F3})$$

Positive FAA = approach/positive affect (common in experienced meditators). Also computes **theta hemispheric asymmetry**.

In [ ]:
def band_power_channel(signal, lo, hi, fs):
    """Mean PSD power for a single channel in a frequency band."""
    freqs, psd = welch(signal, fs=fs, nperseg=fs)
    mask = (freqs >= lo) & (freqs <= hi)
    return psd[mask].mean() if mask.sum() > 0 else 0.0


def asymmetry_features(epoch_data, ch_names, fs):
    """
    Frontal Alpha Asymmetry + Theta Hemispheric Asymmetry.
    Falls back to first/second halves of channels if specific
    electrode names (F3, F4, etc.) are not found.
    """
    feats = {}

    # ── Frontal Alpha Asymmetry ─────────────────────────────────────────
    left_frontal  = ['F3', 'F7', 'AF3', 'Fp1']
    right_frontal = ['F4', 'F8', 'AF4', 'Fp2']

    left_idx  = [i for i, n in enumerate(ch_names) if n in left_frontal]
    right_idx = [i for i, n in enumerate(ch_names) if n in right_frontal]

    if left_idx and right_idx:
        left_alpha  = np.mean([band_power_channel(epoch_data[i], 8, 12, fs) for i in left_idx])
        right_alpha = np.mean([band_power_channel(epoch_data[i], 8, 12, fs) for i in right_idx])
        feats['FAA'] = np.log(right_alpha + 1e-10) - np.log(left_alpha + 1e-10)
    else:
        # Fallback: left half vs right half of channels
        n_ch   = epoch_data.shape[0]
        l_half = epoch_data[:n_ch//2]
        r_half = epoch_data[n_ch//2:]
        l_a    = np.mean([band_power_channel(l_half[i], 8, 12, fs) for i in range(len(l_half))])
        r_a    = np.mean([band_power_channel(r_half[i], 8, 12, fs) for i in range(len(r_half))])
        feats['FAA'] = np.log(r_a + 1e-10) - np.log(l_a + 1e-10)

    # ── Theta Hemispheric Asymmetry ─────────────────────────────────────
    n_ch   = epoch_data.shape[0]
    l_half = epoch_data[:n_ch//2]
    r_half = epoch_data[n_ch//2:]
    l_t    = np.mean([band_power_channel(l_half[i], 4, 8, fs) for i in range(len(l_half))])
    r_t    = np.mean([band_power_channel(r_half[i], 4, 8, fs) for i in range(len(r_half))])
    feats['Theta_Asymmetry'] = np.log(r_t + 1e-10) - np.log(l_t + 1e-10)

    return feats

print('✅ Asymmetry functions defined.')

## 📊 Feature Block 6: PSD Band Powers + Ratios

Same as the original but now includes engineered ratios.

In [ ]:
def psd_features(epoch_data, fs):
    """PSD band powers averaged over all channels + engineered ratios."""
    feats = {}
    for band, (lo, hi) in FREQ_BANDS.items():
        powers = [band_power_channel(epoch_data[ch], lo, hi, fs)
                  for ch in range(epoch_data.shape[0])]
        feats[band] = np.mean(powers)

    eps = 1e-10
    feats['Theta_Beta_ratio']  = feats['Theta']  / (feats['Beta']  + eps)
    feats['Alpha_Beta_ratio']  = feats['Alpha']  / (feats['Beta']  + eps)
    feats['Gamma_Beta_ratio']  = feats['Gamma']  / (feats['Beta']  + eps)
    feats['Theta_Alpha_ratio'] = feats['Theta']  / (feats['Alpha'] + eps)
    return feats

print('✅ PSD + ratio features defined.')

## 🚀 Run All Feature Extraction

In [ ]:
def extract_all_features(subject, task, fs=128):
    """
    Extract complete feature set for all epochs of one subject-task.
    Returns list of dicts (one per epoch).
    """
    ep_file = OUTPUT_PATH / f'{subject}_task-{task}_epochs.fif'
    if not ep_file.exists():
        return []

    epochs   = mne.read_epochs(str(ep_file), preload=True, verbose=False)
    ch_names = epochs.ch_names
    data     = epochs.get_data()  # (n_epochs, n_channels, n_times)
    rows     = []

    for ep_idx in range(len(data)):
        ep = data[ep_idx]  # (n_channels, n_times)
        row = {'Subject': subject, 'Task': task}

        row.update(psd_features(ep, fs))
        row.update(hjorth_features(ep))
        row.update(complexity_features(ep))
        row.update(asymmetry_features(ep, ch_names, fs))
        row.update(plv_features(ep, ch_names, fs))
        row.update(coherence_features(ep, fs))
        rows.append(row)

    del epochs
    gc.collect()
    return rows


print('⏳ Extracting comprehensive features (this may take ~10-20 min)...')
all_rows = []

results = joblib.Parallel(n_jobs=4, verbose=10)(
    joblib.delayed(extract_all_features)(subj, task)
    for subj in subjects
    for task in tasks
)

for r in results:
    all_rows.extend(r)

df = pd.DataFrame(all_rows)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

df.to_csv(OUTPUT_PATH / 'comprehensive_features.csv', index=False)
print(f'\n✅ Done! Shape: {df.shape}')
print(f'   Columns: {list(df.columns)}')

## 📊 Feature Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Load split info
import json
with open(OUTPUT_PATH / 'subject_split.json') as f:
    split_info = json.load(f)

df['Split'] = df['Subject'].apply(lambda s: 'test' if s in split_info['test'] else 'train')

# ── 1. Feature correlation heatmap ──────────────────────────────────────
feature_cols = [c for c in df.columns if c not in ['Subject', 'Task', 'Split']]
fig, ax = plt.subplots(figsize=(14, 12))
corr = df[feature_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            ax=ax, linewidths=0.3, annot=False)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH / 'feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 2. Key features by task ─────────────────────────────────────────────
key_features = ['Theta', 'Alpha', 'FAA', 'SampleEntropy_mean',
                'PLV_Alpha_mean', 'Hjorth_Complexity_mean']
key_features = [f for f in key_features if f in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
palette    = {'med1breath': '#2196F3', 'med2': '#4CAF50', 'think1': '#FF9800', 'think2': '#E91E63'}
for ax, feat in zip(axes.flat, key_features):
    sns.violinplot(data=df, x='Task', y=feat, palette=palette,
                   order=tasks, ax=ax, inner='quartile')
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Key Feature Distributions by Mental State', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / 'features_by_task.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualizations saved.')

## 🤖 Quick Baseline: Random Forest on Comprehensive Features

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# Subject-level split
train_df = df[df['Split'] == 'train'].copy()
test_df  = df[df['Split'] == 'test'].copy()

le = LabelEncoder()
y_train = le.fit_transform(train_df['Task'])
y_test  = le.transform(test_df['Task'])

X_train = train_df[feature_cols].values
X_test  = test_df[feature_cols].values

# Scale (fit on train only)
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

rf = RandomForestClassifier(n_estimators=200, max_depth=20,
                            n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(f'🏆 Random Forest Accuracy (comprehensive features): {accuracy_score(y_test, y_pred)*100:.2f}%')
print('\n', classification_report(y_test, y_pred, target_names=le.classes_))

# Feature importance
imp_df = pd.DataFrame({'Feature': feature_cols, 'Importance': rf.feature_importances_})
imp_df = imp_df.sort_values('Importance', ascending=False).head(20)

plt.figure(figsize=(10, 6))
plt.barh(imp_df['Feature'], imp_df['Importance'], color='steelblue')
plt.xlabel('Importance')
plt.title('Top 20 Most Important Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_PATH / 'feature_importance_comprehensive.png', dpi=150)
plt.show()

---
## ✅ Summary — New Features Added

| Feature Group | # Features | Research Basis |
|---|---|---|
| PSD Bands | 5 | Standard neuroscience |
| Band Ratios | 4 | Theta/Beta = mindfulness marker |
| PLV Connectivity | 9 | Phase synchrony, inter-region comm. |
| Coherence | 6 | Freq-domain coupling |
| Sample/Perm Entropy | 4 | Complexity in meditation |
| Higuchi FD | 1 | Fractal complexity |
| Hjorth | 5 | Activity, mobility, complexity |
| Asymmetry | 2 | FAA = validated meditation marker |

➡️ **Next:** Run `03_deep_learning.ipynb` for EEGNet, CNN-LSTM, Conformer.